In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import time
from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}"

In [ ]:
import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax

---

In [ ]:
from dmpe.utils.density_estimation import build_grid, DensityEstimate
from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from admpe.vector_fields import VectorField, DynamicsVectorField, CurvatureVectorField
from admpe.adaptive_densities import TargetDensity

In [ ]:
env, penalty_function, featurize, env_params = setup_pendulum_env()

In [ ]:
env

In [ ]:
obs, state = env.reset(rng=jax.random.key(12), deterministic_state=True)

In [ ]:
from dmpe.models.models import NeuralEulerODE, NeuralEulerODEPendulum, NeuralEulerODECartpole
from dmpe.algorithms.algorithm_utils import default_dmpe_parameterization
from dmpe.algorithms.algorithms import excite_with_dmpe


from dmpe.evaluation.plotting_utils import plot_feature_combinations, plot_sequence

In [ ]:
points_per_dim = 20

exp_params, proposed_actions, loader_key, expl_key, process_noise_key = default_dmpe_parameterization(
    env,
    seed=12,
    featurize=featurize,
    model_class=NeuralEulerODEPendulum,
    n_time_steps=10000,
    points_per_dim=points_per_dim,
)

# uniform_density = DensityEstimate.from_estimate(
#     p=exp_params["alg_params"]["target_distribution"],
#     n_additional_observations=0,
#     density_estimate=target_density
# )

observations, actions, model, density_estimate, losses, proposed_actions, callback_out = excite_with_dmpe(
    env,
    exp_params,
    proposed_actions,
    loader_key,
    expl_key,
    process_noise_key
)

data_exp_uniform = dict(
    observations=observations,
    actions=actions,
    model=model,
    density_estimate=density_estimate,
    losses=losses,
    proposed_actions=proposed_actions,
)

In [ ]:
print("result")
data_exp_uniform["density_estimate"].visualize(labels=list(env.obs_description) + list(env.action_description))
plt.show()

plot_feature_combinations(jnp.hstack([
    data_exp_uniform["observations"][:-1],
    data_exp_uniform["actions"]
]), list(env.obs_description) + list(env.action_description));
plt.show()

In [ ]:
plot_sequence(
    data_exp_uniform["observations"][:-1],
    data_exp_uniform["actions"],
    env.tau,
    env.obs_description,
    env.action_description,
)

In [ ]:
plot_sequence(
    data_exp_uniform["observations"][1800:2000],
    data_exp_uniform["actions"][1800:2000],
    env.tau,
    env.obs_description,
    env.action_description,
)